# House Prices - Random Forest Regression
**Objetivo:** Predecir el precio de venta de casas usando un modelo Random Forest.
Al final se genera el archivo `submission.csv` listo para subir a Kaggle.

## 1. Importar librerías

In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings('ignore')

print('Librerías importadas correctamente')

Librerías importadas correctamente


## 2. Cargar los datos

In [2]:
train = pd.read_csv('train.csv')
test  = pd.read_csv('test.csv')
sample_submission = pd.read_csv('sample_submission.csv')

print(f'Train: {train.shape}')
print(f'Test:  {test.shape}')
print(f'Sample submission:\n{sample_submission.head()}')

Train: (1460, 81)
Test:  (1459, 80)
Sample submission:
     Id      SalePrice
0  1461  169277.052498
1  1462  187758.393989
2  1463  183583.683570
3  1464  179317.477511
4  1465  150730.079977


## 3. Exploración rápida

In [3]:
train.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [4]:
train.describe()

,Id,MSSubClass,LotFrontage,LotArea,OverallQual,OverallCond,YearBuilt,YearRemodAdd,MasVnrArea,BsmtFinSF1,...,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,MiscVal,MoSold,YrSold,SalePrice
count,1460.000000,1460.000000,1201.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1452.000000,1460.000000,...,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000
mean,730.500000,56.897260,70.049958,10516.828082,6.099315,5.575342,1971.267808,1984.865753,103.685262,443.639726,...,94.244521,46.660274,21.954110,3.409589,15.060959,2.758904,43.489041,6.321918,2007.815753,180921.195890
std,421.610009,42.300571,24.284752,9981.264932,1.382997,1.112799,30.202904,20.645407,181.066207,456.098091,...,125.338794,66.256028,61.119149,29.317331,55.757415,40.177307,496.123024,2.703626,1.328095,79442.502883
min,1.000000,20.000000,21.000000,1300.000000,1.000000,1.000000,1872.000000,1950.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,2006.000000,34900.000000
25%,365.750000,20.000000,59.000000,7553.500000,5.000000,5.000000,1954.000000,1967.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,5.000000,2007.000000,129975.000000
50%,730.500000,50.000000,69.000000,9478.500000,6.000000,5.000000,1973.000000,1994.000000,0.000000,383.500000,...,0.000000,25.000000,0.000000,0.000000,0.000000,0.000000,0.000000,6.000000,2008.000000,163000.000000
75%,1095.250000,70.000000,80.000000,11601.500000,7.000000,6.000000,2000.000000,2004.000000,166.000000,712.250000,...,168.000000,68.000000,0.000000,0.000000,0.000000,0.000000,0.000000,8.000000,2009.000000,214000.000000
max,1460.000000,190.000000,313.000000,215245.000000,10.000000,9.000000,2010.000000,2010.000000,1600.000000,5644.000000,...,857.000000,547.000000,552.000000,508.000000,480.000000,738.000000,15500.000000,12.000000,2010.000000,755000.000000


In [5]:
# Revisar valores nulos en train
nulos_train = train.isnull().sum()
print('Columnas con nulos en TRAIN:')
print(nulos_train[nulos_train > 0].sort_values(ascending=False))

Columnas con nulos en TRAIN:
PoolQC          1453
MiscFeature     1406
Alley           1369
Fence           1179
MasVnrType       872
FireplaceQu      690
LotFrontage      259
GarageType        81
GarageYrBlt       81
GarageFinish      81
GarageQual        81
GarageCond        81
BsmtExposure      38
BsmtFinType2      38
BsmtQual          37
BsmtCond          37
BsmtFinType1      37
MasVnrArea         8
Electrical         1
dtype: int64


## 4. Preprocesamiento

In [6]:
# Separar target y guardar IDs del test
y_train = train['SalePrice']
test_ids = test['Id']

# Unir train y test para hacer el mismo preprocesamiento
train_proc = train.drop(columns=['Id', 'SalePrice'])
test_proc  = test.drop(columns=['Id'])

combined = pd.concat([train_proc, test_proc], axis=0, ignore_index=True)
print(f'Combined shape: {combined.shape}')

Combined shape: (2919, 79)


In [7]:
# Imputar valores nulos
#   - Numéricas  -> mediana
#   - Categóricas -> 'Missing'

num_cols = combined.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = combined.select_dtypes(include=['object']).columns.tolist()

combined[num_cols] = combined[num_cols].fillna(combined[num_cols].median())
combined[cat_cols] = combined[cat_cols].fillna('Missing')

print('Nulos restantes:', combined.isnull().sum().sum())

Nulos restantes: 0


In [8]:
# Codificación one-hot de variables categóricas
combined_encoded = pd.get_dummies(combined, columns=cat_cols, drop_first=False)
print(f'Shape tras one-hot encoding: {combined_encoded.shape}')

Shape tras one-hot encoding: (2919, 310)


In [9]:
# Separar de nuevo en train y test
n_train = train_proc.shape[0]

X_train = combined_encoded.iloc[:n_train].reset_index(drop=True)
X_test  = combined_encoded.iloc[n_train:].reset_index(drop=True)

print(f'X_train: {X_train.shape}')
print(f'X_test:  {X_test.shape}')

X_train: (1460, 310)
X_test:  (1459, 310)


## 5. Modelo Random Forest

In [10]:
# Definir el modelo
rf_model = RandomForestRegressor(
    n_estimators=500,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features='sqrt',
    random_state=42,
    n_jobs=-1
)

# Validación cruzada (5 folds) — métrica: RMSE en escala logarítmica (igual que Kaggle)
cv_scores = cross_val_score(
    rf_model, X_train, np.log1p(y_train),
    cv=5,
    scoring='neg_mean_squared_error',
    n_jobs=-1
)

rmse_cv = np.sqrt(-cv_scores)
print(f'RMSE (log) por fold: {rmse_cv.round(4)}')
print(f'RMSE (log) promedio: {rmse_cv.mean():.4f} ± {rmse_cv.std():.4f}')

RMSE (log) por fold: [0.1292 0.1611 0.1556 0.1376 0.145 ]
RMSE (log) promedio: 0.1457 ± 0.0116


In [11]:
# Entrenar con todos los datos de entrenamiento
rf_model.fit(X_train, np.log1p(y_train))
print('Modelo entrenado correctamente')

Modelo entrenado correctamente


In [12]:
# Importancia de las 20 variables más relevantes
importancias = pd.Series(rf_model.feature_importances_, index=X_train.columns)
print('Top 20 variables más importantes:')
print(importancias.sort_values(ascending=False).head(20))

Top 20 variables más importantes:
OverallQual            0.068928
GrLivArea              0.067243
1stFlrSF               0.042325
GarageArea             0.040215
YearBuilt              0.040182
GarageCars             0.039146
TotalBsmtSF            0.037595
FullBath               0.032263
ExterQual_TA           0.029491
YearRemodAdd           0.026641
GarageYrBlt            0.026586
KitchenQual_TA         0.026349
Fireplaces             0.020423
LotArea                0.019159
FireplaceQu_Missing    0.019100
TotRmsAbvGrd           0.018710
BsmtFinSF1             0.017844
2ndFlrSF               0.017056
ExterQual_Gd           0.015961
OpenPorchSF            0.014242
dtype: float64


## 6. Predicciones sobre el conjunto de test

In [13]:
# Predecir (el modelo predice log1p(precio), revertimos con expm1)
y_pred_log = rf_model.predict(X_test)
y_pred = np.expm1(y_pred_log)

print(f'Predicciones generadas: {len(y_pred)}')
print(f'Precio mínimo predicho: ${y_pred.min():,.0f}')
print(f'Precio máximo predicho: ${y_pred.max():,.0f}')
print(f'Precio promedio predicho: ${y_pred.mean():,.0f}')

Predicciones generadas: 1459
Precio mínimo predicho: $65,056
Precio máximo predicho: $427,476
Precio promedio predicho: $175,037


## 7. Generar archivo de submission para Kaggle

In [14]:
# Construir submission con el mismo formato que sample_submission.csv
submission = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': y_pred
})

# Verificar que coincide con sample_submission
print('Columnas submission:', submission.columns.tolist())
print(f'Filas: {len(submission)}')
print(submission.head(10))

Columnas submission: ['Id', 'SalePrice']
Filas: 1459
     Id      SalePrice
0  1461  124201.160785
1  1462  150939.288810
2  1463  180277.854045
3  1464  190686.558703
4  1465  187248.760916
5  1466  182072.659632
6  1467  168751.664615
7  1468  175485.026832
8  1469  179725.780952
9  1470  130125.212095


In [15]:
# Guardar el archivo
submission.to_csv('submission.csv', index=False)
print('Archivo submission.csv generado correctamente')
print(f'Listo para subir a Kaggle!')

Archivo submission.csv generado correctamente
Listo para subir a Kaggle!
